In [ ]:
!pip install groq -q
print('libraries installed sucessfully!!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.4 MB/s eta 0:00:00
libraries installed sucessfully!!


import all required libraries

In [ ]:
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print('libraries installed sucessfully!!')

libraries installed sucessfully!!


initialising AI model

In [ ]:
import os
os.environ["GROQ_API_KEY"]="---------------------------------------------------------------"
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"
print("Groq client created sucessfully")
print("uisng model",MODEL)


Groq client created sucessfully
uisng model llama-3.1-8b-instant


In [ ]:
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""





df=pd.read_csv(io.StringIO(csv_data))
print('dataset loaded:',len(df))
print('row:',df.shape[0])
print('columns:',df.shape[1])
df.head()

dataset loaded: 30
row: 30
columns: 8


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


creating sqlite database

In [ ]:
conn=sqlite3.connect('college.db')
df.to_sql('students',conn,if_exists='replace',index=False)
print('db created ')
test_df=pd.read_sql_query('select COUNT(*) as tottal_rows  from students',conn)
print(test_df)

db created 
   tottal_rows
0           30


In [ ]:
def get_schema(conn,table_name="students"):
  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns=cursor.fetchall()
  schema_lines=[f"Table: {table_name}"]
  for col in columns:
    schema_lines.append(f"  - {col[1]} ({col[2]})")
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows=cursor.fetchall()
  schema_lines.append("\nSample Rows  (first 3):")
  for row in sample_rows:
    schema_lines.append(f"  - {row}")
  return "\n".join(schema_lines)
schema=get_schema(conn)
print(schema)


Table: students
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - subject (TEXT)
  - marks (INTEGER)
  - attendance (INTEGER)
  - grade (TEXT)

Sample Rows  (first 3):
  - (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
  - (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
  - (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [14]:
def generate_sql(user_question,schema_text,client,model):
    system_prompt = f"""You are an expert SQL assistant.
You are connected to a SQLite database with the following structure:


{schema_text}


Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text — only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is: students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question}
        ],
        temperature=0.0
    )

    sql_query=response.choices[0].message.content.strip()
    return sql_query
question='show me all male students'
print('Question:',question)
sql=generate_sql(question,schema,client,MODEL)
print('SQL:',sql)




Question: show me all male students
SQL: SELECT * FROM students WHERE gender = 'Male'


In [ ]:
import re

def execute_sql(sql_query, conn):
    clean_sql = sql_query.strip()

    clean_sql = re.sub(r'^```sql\s*', '', clean_sql)
    clean_sql = re.sub(r'^```\s*', '', clean_sql)
    clean_sql = clean_sql.strip()
    try:
      result_df=pd.read_sql_query(clean_sql,conn)
      return result_df,None
    except Exception as e:
      return None,str(e)
print('sql:',sql)
result,error=execute_sql(sql,conn)
if error:
  print('error:',error)
else:
  print('result:',result)


sql: SELECT * FROM students WHERE gender = 'Male'
result:     student_id           name  age gender      subject  marks  attendance  \
0            1   Aarav Sharma   20   Male  Mathematics     88          92   
1            3    Rohan Mehta   20   Male  Programming     95          98   
2            5     Arjun Nair   21   Male  Programming     91          94   
3            7    Karan Singh   22   Male  Mathematics     74          81   
4            9   Vikram Reddy   20   Male      Science     70          79   
5           11   Aditya Kumar   21   Male  Programming     97          99   
6           13    Rahul Desai   22   Male  Mathematics     68          80   
7           15   Nikhil Verma   20   Male      Science     77          84   
8           17   Manish Joshi   21   Male  Programming     73          82   
9           19    Suresh Babu   22   Male  Mathematics     82          89   
10          21    Deepak Nair   20   Male      Science     79          86   
11          23   S

In [ ]:
print(result.head())

   student_id          name  age gender      subject  marks  attendance grade
0           1  Aarav Sharma   20   Male  Mathematics     88          92     A
1           3   Rohan Mehta   20   Male  Programming     95          98    A+
2           5    Arjun Nair   21   Male  Programming     91          94    A+
3           7   Karan Singh   22   Male  Mathematics     74          81     B
4           9  Vikram Reddy   20   Male      Science     70          79     B


In [21]:
def text_to_sql(user_question, conn, client, model, verbose=True):
    print('User Question:', user_question)
    if verbose:
        print('Generating schema...')
    schema_text = get_schema(conn)
    if verbose:
        print('Schema loaded successfully.')
    if verbose:
        print('Generating SQL...')
    generated_sql = generate_sql(user_question,schema_text,client,model)
    if verbose:
        print('Generated SQL:')
        print(generated_sql)
    if verbose:
        print('Executing SQL...')
    result_df, error = execute_sql(generated_sql, conn)
    if error:
        print('Error:', error)
        return None, generated_sql
    if verbose:
        print('Results:')
        print(result_df.to_string(index=False))
    return result_df, generated_sql
result,sql_used=text_to_sql('show top 5 students in programming',conn,client,MODEL)



User Question: show top 5 students in programming
Generating schema...
Schema loaded successfully.
Generating SQL...
Generated SQL:
SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
Executing SQL...
Results:
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+


In [27]:
result,sql_used=text_to_sql('show the students who are having age greater than 21',conn,client,MODEL)

User Question: show the students who are having age greater than 21
Generating schema...
Schema loaded successfully.
Generating SQL...
Generated SQL:
SELECT * FROM students WHERE age > 21
Executing SQL...
Results:
 student_id          name  age gender     subject  marks  attendance grade
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Female Mathematics     58          73     D
         25   Vijay Kumar   22   Male Mathematics     71          83     B
         28  Sunita Gupta   22 Female Mathematics     87          93     A


In [24]:
result,sql_used=text_to_sql('show the number of students in maths and their marks',conn,client,MODEL)

User Question: show the number of students in maths and their marks
Generating schema...
Schema loaded successfully.
Generating SQL...
Generated SQL:
SELECT COUNT(student_id), marks FROM students WHERE subject = 'Mathematics'
Executing SQL...
Results:
 COUNT(student_id)  marks
                10     88


In [25]:
result,sql_used=text_to_sql('show tthe students who all got A+ grade with their names',conn,client,MODEL)

User Question: show tthe students who all got A+ grade with their names
Generating schema...
Schema loaded successfully.
Generating SQL...
Generated SQL:
SELECT name FROM students WHERE grade = 'A+'
Executing SQL...
Results:
         name
  Rohan Mehta
   Arjun Nair
 Aditya Kumar
Swathi Pillai
 Anjali Singh
  Nandita Rao
 Bhavna Mehta


In [28]:
result,sql_used=text_to_sql('Attendance 75 ku keela iruka students ah kaatu',conn,client,MODEL)

User Question: Attendance 75 ku keela iruka students ah kaatu
Generating schema...
Schema loaded successfully.
Generating SQL...
Generated SQL:
SELECT * FROM students WHERE attendance < 75
Executing SQL...
Results:
 student_id         name  age gender     subject  marks  attendance grade
         10 Pooja Sharma   22 Female Mathematics     55          72     D
         22 Rekha Sharma   22 Female Mathematics     58          73     D
